# Extract 
- From Azure Blob storage

In [104]:
import os
from io import BytesIO

import pandas as pd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

from datetime import date

load_dotenv()
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

# Load function
def load(container_name, file_path):
    """Read one CSV file from Azure Blob Storage and return it as a DataFrame."""
    if not connection_string:
        raise ValueError(
            "AZURE_STORAGE_CONNECTION_STRING is not set in the environment."
        )

    print(f"Loading: {file_path}")

    blob_service_client = BlobServiceClient.from_connection_string(
        connection_string
    )
    container_client = blob_service_client.get_container_client(container_name)

    blob_client = container_client.get_blob_client(file_path)
    blob_data = blob_client.download_blob().readall()

    df = pd.read_csv(BytesIO(blob_data))
    print(f"File successful: {file_path}")
    return df

Extract files(container: baraa)
- source_crm
    - cust_info.csv
    - prd_info.csv
    - sales_details.csv
- source_erp
    - CUST_AZ12.csv
    - LOC_A101.csv
    - PX_CAT_G1V2.csv

In [105]:
def load_all():
    try:
        global crm_cust_info, crm_prd_info, crm_sales_details
        global erp_cust_az12, erp_loc_a101, erp_px_cat_g1v2

        crm_cust_info = load("baraa", "source_crm/cust_info.csv")
        crm_prd_info = load("baraa", "source_crm/prd_info.csv")
        crm_sales_details = load("baraa", "source_crm/sales_details.csv")
        erp_cust_az12 = load("baraa", "source_erp/CUST_AZ12.csv")
        erp_loc_a101 = load("baraa", "source_erp/LOC_A101.csv")
        erp_px_cat_g1v2 = load("baraa", "source_erp/PX_CAT_G1V2.csv")

        print("All csv files are loaded")
        return {
            "crm_cust_info": crm_cust_info,
            "crm_prd_info": crm_prd_info,
            "crm_sales_details": crm_sales_details,
            "erp_cust_az12": erp_cust_az12,
            "erp_loc_a101": erp_loc_a101,
            "erp_px_cat_g1v2": erp_px_cat_g1v2,
        }
    except Exception as e:
        print("File not completely loaded:", e)
        return None

all_data = load_all()

# Example access:
# crm_cust_info.head()
# crm_prd_info.head()

Loading: source_crm/cust_info.csv
File successful: source_crm/cust_info.csv
Loading: source_crm/prd_info.csv
File successful: source_crm/prd_info.csv
Loading: source_crm/sales_details.csv
File successful: source_crm/sales_details.csv
Loading: source_erp/CUST_AZ12.csv
File successful: source_erp/CUST_AZ12.csv
Loading: source_erp/LOC_A101.csv
File successful: source_erp/LOC_A101.csv
Loading: source_erp/PX_CAT_G1V2.csv
File successful: source_erp/PX_CAT_G1V2.csv
All csv files are loaded


# Transform
- cleaning and standardize the data

In [106]:
def clean_customer_data(crm_cust_info):
    """Clean and transform customer data."""

    # 1. Copy source DataFrame
    df = crm_cust_info.copy()

    # 2. Remove duplicates and rows missing required IDs
    df = df.drop_duplicates()
    df = df.dropna(subset=["cst_id", "cst_key"])

    # 3. Convert data types
    df["cst_id"] = df["cst_id"].astype(int)

    df["cst_create_date"] = pd.to_datetime(
        df["cst_create_date"],
        errors="coerce"
    )

    # 4. Keep the latest record for each customer
    df = (
        df.sort_values("cst_create_date")
          .drop_duplicates(
              subset="cst_id",
              keep="last"
          )
    )

    # 5. Clean customer names
    name_columns = [
        "cst_firstname2",
        "cst_lastname2"
    ]

    df[name_columns] = df[name_columns].apply(
        lambda col: col.str.strip()
    )

    # 6. Rename columns
    df = df.rename(columns={
        "cst_firstname2": "cst_firstname",
        "cst_lastname2": "cst_lastname"
    })

    # 7. Convert coded values to readable values
    df["cst_marital_status"] = df["cst_marital_status"].replace({
        "M": "Married",
        "S": "Single"
    })

    df["cst_gndr"] = df["cst_gndr"].replace({
        "M": "Male",
        "F": "Female"
    })

    # 8. Replace remaining missing values
    df = df.fillna("N/A")

    # 9. Return cleaned DataFrame
    return df

In [107]:
df_customer = clean_customer_data(crm_cust_info)

In [108]:
# crm_prd_info transformation
df= crm_prd_info
# drop unnecessary
df=df.drop_duplicates()
df=df.dropna(subset=["prd_id","prd_key"])


In [109]:
# check data
print("\n Information")
print(df.info())
print("\n Number of NULL data")
print(df.isnull().sum())
print("\n Number of unique data")
print(df.nunique())


 Information
<class 'pandas.DataFrame'>
RangeIndex: 397 entries, 0 to 396
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   prd_id        397 non-null    int64  
 1   prd_key       397 non-null    str    
 2   prd_nm        397 non-null    str    
 3   prd_cost      395 non-null    float64
 4   prd_line      380 non-null    str    
 5   prd_start_dt  397 non-null    str    
 6   prd_end_dt    200 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 21.8 KB
None

 Number of NULL data
prd_id            0
prd_key           0
prd_nm            0
prd_cost          2
prd_line         17
prd_start_dt      0
prd_end_dt      197
dtype: int64

 Number of unique data
prd_id          397
prd_key         295
prd_nm          295
prd_cost        108
prd_line          4
prd_start_dt      4
prd_end_dt        2
dtype: int64


In [110]:
# check is null
df[df["cst_gndr"].isna()]

KeyError: 'cst_gndr'

In [ ]:
df.head(10)

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,NaN,R,2003-07-01,NaN
1,211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,NaN,R,2003-07-01,NaN
2,212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12.0,S,2011-07-01,2007-12-28
3,213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14.0,S,2012-07-01,2008-12-27
4,214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13.0,S,2013-07-01,NaN
5,215,AC-HE-HL-U509,Sport-100 Helmet- Black,12.0,S,2011-07-01,2007-12-28
6,216,AC-HE-HL-U509,Sport-100 Helmet- Black,14.0,S,2012-07-01,2008-12-27
7,217,AC-HE-HL-U509,Sport-100 Helmet- Black,13.0,S,2013-07-01,NaN
8,218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3.0,M,2011-07-01,2007-12-28
9,219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3.0,M,2011-07-01,2007-12-28


In [ ]:
df.tail(10)

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
387,597,BI-MB-BK-M18B-42,Mountain-500 Black- 42,295.0,M,2013-07-01,NaN
388,598,BI-MB-BK-M18B-44,Mountain-500 Black- 44,295.0,M,2013-07-01,NaN
389,599,BI-MB-BK-M18B-48,Mountain-500 Black- 48,295.0,M,2013-07-01,NaN
390,600,BI-MB-BK-M18B-52,Mountain-500 Black- 52,295.0,M,2013-07-01,NaN
391,601,CO-BB-BB-7421,LL Bottom Bracket,24.0,NaN,2013-07-01,NaN
392,602,CO-BB-BB-8107,ML Bottom Bracket,45.0,NaN,2013-07-01,NaN
393,603,CO-BB-BB-9108,HL Bottom Bracket,54.0,NaN,2013-07-01,NaN
394,604,BI-RB-BK-R19B-44,Road-750 Black- 44,344.0,R,2013-07-01,NaN
395,605,BI-RB-BK-R19B-48,Road-750 Black- 48,344.0,R,2013-07-01,NaN
396,606,BI-RB-BK-R19B-52,Road-750 Black- 52,344.0,R,2013-07-01,NaN


# Load
- loaded into a MySQL datawarehouse